# Part 2, Topic 2: Voltage Glitching to Bypass Password

---
NOTE: This lab references some (commercial) training material on [ChipWhisperer.io](https://www.ChipWhisperer.io). You can freely execute and use the lab per the open-source license (including using it in your own courses if you distribute similarly), but you must maintain notice about this source location. Consider joining our training course to enjoy the full experience.

---

**SUMMARY:** *We've seen how voltage glitching can be used to corrupt calculations, just like clock glitching. Let's continue on and see if it can also be used to break past a password check.*

**LEARNING OUTCOMES:**

* Applying previous glitch settings to new firmware
* Checking for success and failure when glitching

## Firmware

Again, we've already covered this lab, so it'll be mostly up to you!

In [1]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CW312_SAM4S'
SS_VER = 'SS_VER_2_1'

In [15]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

INFO: Caught exception on reconnecting to target - attempting to reconnect to scope first.
INFO: This is a work-around when USB has died without Python knowing. Ignore errors above this line.
INFO: Found ChipWhisperer😍
scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 346246                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.io.tio1     

In [3]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../../firmware/mcu/simpleserial-glitch
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

SS_VER set to SS_VER_2_1
SS_VER set to SS_VER_2_1
arm-none-eabi-gcc (15:14.2.rel1-1) 14.2.1 20241119
Copyright (C) 2024 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CW312_SAM4S 
.
Welcome to another exciting ChipWhisperer target build!!
.
.
.
Compiling:
Compiling:
Compiling:
.
-en     simpleserial-glitch.c ...
-en     .././simpleserial/simpleserial.c ...
-en     .././hal/hal.c ...
Compiling:
-en     .././hal//sam4s/startup_sam4s.c ...
.
.
.
Compiling:
Compiling:
Compiling:
.
-en     .././hal//sam4s/sam4s_hal.c ...
-en     .././hal//sam4s/uart.c ...
-en     .././hal//sam4s/pio.c ...
Compiling:
-en     .././hal//sam4s/system_sam4s.c ...
.
Compiling:
-en     .././hal//sam4s/sysclk.c ...
.
Compiling:
-en     .././hal//sam4s/pmc.c ...
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
.
LINKING:
-en  

In [16]:
fw_path = "../../../firmware/mcu/simpleserial-glitch/simpleserial-glitch-{}.hex".format(PLATFORM)
cw.program_target(scope, prog, fw_path)
if SS_VER=="SS_VER_2_1":
    target.reset_comms()

In [60]:
def reboot_flush():
    reset_target(scope)
    target.flush()
    

print("target.baud ", target.baud)
print("scope.clock.clkgen_freq ", scope.clock.clkgen_freq)
# target.baud  230400
# scope.clock.clkgen_freq  7363636.363636363

target.baud  230400
scope.clock.clkgen_freq  7363636.363636363


In [42]:
#Do glitch loop
reboot_flush()
pw = bytearray([0x74, 0x6F, 0x75, 0x63, 0x68]) # valid pw
pw = bytearray([0x70, 0x6F, 0x75, 0x63, 0x68]) # wrong pw
target.simpleserial_write('p', pw)

val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10)#For loop check
valid = val['valid']
if valid:
    response = val['payload']
    raw_serial = val['full_response']
    error_code = val['rv']

print(val)

{'valid': True, 'payload': CWbytearray(b'00'), 'full_response': CWbytearray(b'00 72 01 00 99 00'), 'rv': bytearray(b'\x00')}


Like with clock glitching, the scope object can set some typical glitch settings for you:

In [19]:
if scope._is_husky:
    scope.vglitch_setup('hp', default_setup=False) # HP alone works best for Husky
else:
    scope.vglitch_setup('both', default_setup=False) # use both transistors

In [20]:
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["width", "offset", "ext_offset"])
gc.display_stats()

IntText(value=0, description='success count:', disabled=True)

IntText(value=0, description='reset count:', disabled=True)

IntText(value=0, description='normal count:', disabled=True)

FloatSlider(value=0.0, continuous_update=False, description='width setting:', disabled=True, max=10.0, readout…

FloatSlider(value=0.0, continuous_update=False, description='offset setting:', disabled=True, max=10.0, readou…

FloatSlider(value=0.0, continuous_update=False, description='ext_offset setting:', disabled=True, max=10.0, re…

In [58]:
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None})

Parameter name clashes for keys ['data']

:DynamicMap   []
   :Overlay
      .Points.I  :Points   [width,offset]
      .Points.II :Points   [width,offset]

In [57]:
gc.set_range("ext_offset", 0, 150)
gc.set_range("width", 1600, 1900)
gc.set_range("offset", 2000, 2500)
gc.set_global_step([50])
gc.set_step("ext_offset", 1)
gc.set_step("width", 50)
gc.set_step("ext_offset", 1)

scope.glitch.clk_src = "pll"

#scope.glitch.output = "glitch_only" # glitch_out = clk ^ glitch
#scope.glitch.trigger_src = "ext_single" # glitch only after scope.arm() called
#scope.glitch.enabled = True
#scope.io.glitch_hp = False
#scope.io.glitch_hp = True
#scope.io.glitch_lp = False
#scope.io.glitch_lp = False



In [59]:
#disable logging
cw.set_all_log_levels(cw.logging.CRITICAL)

scope.adc.timeout = 0.1
successes = 0

reboot_flush()

for glitch_settings in gc.glitch_values():
    scope.glitch.offset = glitch_settings[1]
    scope.glitch.width = glitch_settings[0]
    scope.glitch.ext_offset = glitch_settings[2]
    if scope.adc.state:
        # can detect crash here (fast) before timing out (slow)
        #print("Trigger still high!")
        gc.add("reset")
        reboot_flush()

    scope.arm()
    target.simpleserial_write('p', bytearray([0]*5))
    ret = scope.capture()
    scope.io.vglitch_reset()
    
    if ret:
        #print('Timeout - no trigger')
        gc.add("reset")

        #Device is slow to boot?
        reboot_flush()
    else:
        val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10, timeout=50)#For loop check
        if val['valid'] is False:
            gc.add("reset")
        else:
            if val['payload'] == bytearray([1]): #for loop check
                successes +=1 
                gc.add("success")
                print(val)
                print(val['payload'])
                print(scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset)
                print("🐙", end="")
            else:
                gc.add("normal")
                     
#reenable logging
cw.set_all_log_levels(cw.logging.WARNING)

In [ ]:

import re
import struct
gc.set_range("ext_offset", 0, 150)
g_step = 0.4
gc.set_global_step(g_step)

#disable logging
cw.set_all_log_levels(cw.logging.CRITICAL)

gc.set_range("width", ???, ???)
gc.set_range("offset", ???, ???)
gc.set_global_step(???)

gc.set_step("ext_offset", 1) # check each clock cycle

scope.adc.timeout = 0.1

reboot_flush()

successes = 0

for glitch_settings in gc.glitch_values():
    scope.glitch.offset = glitch_settings[1]
    scope.glitch.width = glitch_settings[0]
    scope.glitch.ext_offset = glitch_settings[2]
    if scope.adc.state:
        # can detect crash here (fast) before timing out (slow)
        #print("Trigger still high!")
        gc.add("reset")
        reboot_flush()

    scope.arm()
    target.simpleserial_write('p', bytearray([0]*5))
    ret = scope.capture()
    scope.io.vglitch_reset()
    if ret:
        #print('Timeout - no trigger')
        gc.add("reset")

        #Device is slow to boot?
        reboot_flush()
    else:
        val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10, timeout=50)#For loop check
        if ???:
            gc.add("reset")
        else:
            if ???: #for loop check
                successes +=1 
                gc.add("success")
                print(val)
                print(val['payload'])
                print(scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset)
                print("🐙", end="")
            else:
                gc.add("normal")
                    
#reenable logging
cw.set_all_log_levels(cw.logging.WARNING)

Let's see where we needed to target for our glitch to work:

In [24]:
gc.calc(["width", "offset"], "success_rate")

[((150,),
  {'total': 11,
   'success': 0,
   'success_rate': 0.0,
   'reset': 0,
   'reset_rate': 0.0,
   'normal': 11,
   'normal_rate': 1.0}),
 ((149,),
  {'total': 17,
   'success': 0,
   'success_rate': 0.0,
   'reset': 6,
   'reset_rate': 0.35294117647058826,
   'normal': 11,
   'normal_rate': 0.6470588235294118}),
 ((148,),
  {'total': 17,
   'success': 0,
   'success_rate': 0.0,
   'reset': 12,
   'reset_rate': 0.7058823529411765,
   'normal': 5,
   'normal_rate': 0.29411764705882354}),
 ((147,),
  {'total': 17,
   'success': 0,
   'success_rate': 0.0,
   'reset': 12,
   'reset_rate': 0.7058823529411765,
   'normal': 5,
   'normal_rate': 0.29411764705882354}),
 ((146,),
  {'total': 11,
   'success': 0,
   'success_rate': 0.0,
   'reset': 6,
   'reset_rate': 0.5454545454545454,
   'normal': 5,
   'normal_rate': 0.45454545454545453}),
 ((145,),
  {'total': 11,
   'success': 0,
   'success_rate': 0.0,
   'reset': 0,
   'reset_rate': 0.0,
   'normal': 11,
   'normal_rate': 1.0}),
 

In [26]:
results = gc.calc( sort="success_rate")
results

[((1900, 2500, 150),
  {'total': 1,
   'success': 0,
   'success_rate': 0.0,
   'reset': 0,
   'reset_rate': 0.0,
   'normal': 1,
   'normal_rate': 1.0}),
 ((1900, 2500, 149),
  {'total': 2,
   'success': 0,
   'success_rate': 0.0,
   'reset': 1,
   'reset_rate': 0.5,
   'normal': 1,
   'normal_rate': 0.5}),
 ((1900, 2500, 148),
  {'total': 2,
   'success': 0,
   'success_rate': 0.0,
   'reset': 2,
   'reset_rate': 1.0,
   'normal': 0,
   'normal_rate': 0.0}),
 ((1900, 2500, 147),
  {'total': 2,
   'success': 0,
   'success_rate': 0.0,
   'reset': 2,
   'reset_rate': 1.0,
   'normal': 0,
   'normal_rate': 0.0}),
 ((1900, 2500, 146),
  {'total': 1,
   'success': 0,
   'success_rate': 0.0,
   'reset': 1,
   'reset_rate': 1.0,
   'normal': 0,
   'normal_rate': 0.0}),
 ((1900, 2500, 145),
  {'total': 1,
   'success': 0,
   'success_rate': 0.0,
   'reset': 0,
   'reset_rate': 0.0,
   'normal': 1,
   'normal_rate': 1.0}),
 ((1900, 2500, 144),
  {'total': 1,
   'success': 0,
   'success_rate'

In [ ]:
scope.dis()
target.dis()